# SiPeka — Sistem Prediksi Volume Pengangkutan Sampah DKI Jakarta

## Step 6: Modeling & Evaluasi (Split Kronologis 70 / 15 / 15)

**SiPeka** memprediksi volume pengangkutan sampah harian (`m³`) di kali/sungai/situ/waduk DKI Jakarta untuk membantu penjadwalan truk dan alokasi petugas Dinas Lingkungan Hidup.

Notebook ini adalah **Step 6** dari pipeline: membangun, membandingkan, dan mengevaluasi model regresi untuk `volume_sampah_perhari` per `titik_lokasi` per hari menggunakan pembagian waktu **70% train / 15% validation / 15% test**, lalu menyimpan model terbaik untuk dipakai dashboard.

### Ringkasan Tahapan

| Bagian | Isi |
|--------|-----|
| **0** | Setup `SparkSession` dan konstanta eksperimen (`70/15/15`, grid RF & GBT) |
| **1** | Muat & validasi kontrak kolom Parquet `features_v1` dan `features_v2` (output Step 4) |
| **2** | Split kronologis berbasis tanggal: **70% `train` -> 15% `validation` -> 15% `test`** (anti-leakage) |
| **3** | Baseline naif: rata-rata historis per `titik_lokasi` (tanpa outlier ekstrem) |
| **4** | Linear Regression (`StringIndexer -> OneHotEncoder -> StandardScaler`, L2/ElasticNet) |
| **5** | Random Forest & Gradient-Boosted Trees (`GBTRegressor`) dengan `maxBins` otomatis |
| **6** | Eksekusi baseline, LR, RF, dan GBT untuk `v1` dan `v2` |
| **7** | Pemilihan model terbaik (berdasarkan RMSE validation) & penyimpanan artefak |
| **8** | Analisis error (`wilayah`, `bulan`, top-10 titik), visualisasi sampel, & feature importance |
| **9** | Wrapper `PrediktorMasaDepan` untuk `prediksi(tanggal, titik_lokasi)` + uji cepat |
| **10–11** | Interpretasi hasil, checklist checkpoint, dan troubleshooting |

**Cakupan:** Notebook ini tidak membaca `.xls`/CSV mentah, tidak melakukan cleaning/feature engineering (Step 2–4), dan tidak melakukan split acak.

---

### Prasyarat & Urutan Eksekusi

- **Data input:** `data/processed/features_v1` dan `data/processed/features_v2` hasil Step 4 (atau HDFS via `SIPEKA_FEATURES_V1` / `SIPEKA_FEATURES_V2`).
- **Urutan eksekusi:** jalankan sel satu per satu dari atas (Bagian 0–5 definisi fungsi, Bagian 6 training & tuning, Bagian 7–9 penyimpanan artefak, analisis error, dan uji prediksi).
- **Keterbatasan RAM:** `train` difilter `is_outlier == 0` (`<= P99,9`) agar nilai typo ekstrem data mentah tidak merusak model, sedangkan `validation` dan `test` dievaluasi penuh apa adanya.

## 0. Impor & Konfigurasi

Menyiapkan library, path root proyek, `SparkSession`, dan seluruh konstanta eksperimen. Belum ada pembacaan data maupun training.

### 0.1 Impor library & path proyek

In [1]:
from __future__ import annotations

import json
import math
import os
import sys
import time
from datetime import datetime, timedelta
from pathlib import Path

ROOT_PROYEK = Path(os.environ.get("PYTHONPATH", "/workspace"))
if not ROOT_PROYEK.exists():
    ROOT_PROYEK = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT_PROYEK) not in sys.path:
    sys.path.insert(0, str(ROOT_PROYEK))

import pandas as pd
import plotly.express as px
from IPython.display import display
from pyspark import StorageLevel
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import (
    OneHotEncoder,
    StandardScaler,
    StringIndexer,
    VectorAssembler,
)
from pyspark.ml.regression import (
    GBTRegressor,
    LinearRegression,
    RandomForestRegressor,
)
from pyspark.sql import DataFrame, Window
from pyspark.sql import functions as F

from config.settings import (
    DATA_PROCESSED_DIR,
    FIGURES_DIR,
    MODELS_DIR,
    get_settings,
)
from src.utils.spark_session import get_spark, tampilkan_info

### 0.2 SparkSession & konstanta eksperimen

In [2]:
SPARK = get_spark("SiPeka-Step6-Modeling")
tampilkan_info(SPARK)

SETTINGS = get_settings()
SETTINGS.pastikan_folder_lokal()

FEATURES_V1 = os.environ.get(
    "SIPEKA_FEATURES_V1", str(DATA_PROCESSED_DIR / "features_v1")
)
FEATURES_V2 = os.environ.get(
    "SIPEKA_FEATURES_V2", str(DATA_PROCESSED_DIR / "features_v2")
)

TARGET = "volume_sampah_perhari"
KOLOM_TANGGAL = "tanggal"
KOLOM_KATEGORIK = ["titik_lokasi", "kecamatan", "wilayah"]

# Kontrak nama kolom dari Step 4.
FITUR_KALENDER = [
    "tahun",
    "bulan",
    "hari_dalam_bulan",
    "hari_dalam_minggu",
    "is_weekend",
    "minggu_ke",
    "kuartal",
    "hari_ke_dalam_tahun",
    "bulan_sin",
    "bulan_cos",
    "hari_sin",
    "hari_cos",
    "is_musim_hujan",
    "is_hari_libur",
]
FITUR_HISTORIS = [
    "lag_1",
    "lag_7",
    "lag_14",
    "rolling_mean_7",
    "rolling_mean_14",
    "rolling_mean_30",
    "rolling_std_7",
    "rolling_max_7",
    "titik_hist_mean",
    "titik_hist_zero_rate",
]

SEED = 42
RASIO_TRAIN = 0.70
RASIO_VALID = 0.15  # sisa (~0.15) menjadi test
BULAN_MUSIM_HUJAN = {10, 11, 12, 1, 2, 3, 4}

GRID_RF = {
    "numTrees": [30, 50],
    "maxDepth": [6, 8],
    "minInstancesPerNode": [5, 10],
}
GRID_GBT = {
    "maxIter": [30, 50],
    "maxDepth": [4, 6],
    "stepSize": [0.1],
}

MAX_BINS_MINIMUM = 32
SAMPEL_VISUALISASI = 5000
LATIH_TANPA_OUTLIER = True  # kecualikan is_outlier=1 hanya saat melatih model

App Name       : SiPeka-Step6-Modeling
Spark Version  : 3.5.3
Master         : local[4]
Driver Memory  : 3g
Default FS     : file:///


## 1. Kontrak Data & Pemuatan Fitur

Kedua dataset fitur (`v1` dan `v2`) divalidasi terhadap kontrak kolom wajib. Seluruh fitur numerik dan `volume_sampah_perhari` di-cast ke `double`; baris tanpa `titik_lokasi` atau target dibuang.

In [3]:
def path_ke_uri(path: str | Path) -> str:
    """Konversi path lokal ke URI file:// agar Spark tidak mencarinya di HDFS."""
    teks = str(path)
    if "://" in teks:
        return teks
    return Path(teks).resolve().as_uri()


def fitur_numerik_untuk_versi(versi: str) -> list[str]:
    """Daftar fitur numerik wajib untuk versi fitur ('v1' atau 'v2')."""
    if versi not in {"v1", "v2"}:
        raise ValueError(f"Versi fitur tidak dikenal: {versi}")
    fitur = list(FITUR_KALENDER)
    if versi == "v2":
        fitur.extend(FITUR_HISTORIS)
    return fitur


def baca_parquet_fitur(path: str | Path, versi: str) -> DataFrame:
    """Baca Parquet fitur Step 4 dan validasi kontrak kolomnya."""
    uri = path_ke_uri(path)
    try:
        df = SPARK.read.parquet(uri)
    except Exception as exc:
        raise FileNotFoundError(
            f"Dataset fitur {versi} tidak ditemukan atau tidak bisa dibaca: {uri}\n"
            "Jalankan Step 3-4 terlebih dahulu."
        ) from exc

    if KOLOM_TANGGAL not in df.columns:
        raise ValueError(
            f"Kolom '{KOLOM_TANGGAL}' tidak ada di dataset {versi}."
        )

    kolom_waktu: dict[str, F.Column] = {}
    if "tahun" not in df.columns:
        kolom_waktu["tahun"] = F.year(F.col(KOLOM_TANGGAL))
    if "bulan" not in df.columns:
        kolom_waktu["bulan"] = F.month(F.col(KOLOM_TANGGAL)).cast("int")
    else:
        kolom_waktu["bulan"] = F.col("bulan").cast("int")
    df = df.withColumns(kolom_waktu)

    numerik = fitur_numerik_untuk_versi(versi)
    wajib = [TARGET, KOLOM_TANGGAL, "tahun", "bulan"] + KOLOM_KATEGORIK + numerik
    hilang = [c for c in wajib if c not in df.columns]
    if hilang:
        raise ValueError(
            f"Kolom wajib tidak ada di dataset {versi}: {hilang}. "
            "Samakan dengan output Step 4."
        )

    cast_map = {kolom: F.col(kolom).cast("double") for kolom in numerik}
    cast_map[TARGET] = F.col(TARGET).cast("double")
    df = df.withColumns(cast_map)

    return df.filter(
        F.col("titik_lokasi").isNotNull() & F.col(TARGET).isNotNull()
    )

## 2. Split Kronologis 70 / 15 / 15 (Anti-Leakage)

Pembagian dilakukan berbasis batas tanggal sehingga rasio jumlah baris mendekati **70% `train` : 15% `validation` : 15% `test`** dengan urutan waktu ketat:

```text
train (70%)  <  validation (15%)  <  test (15%)
```

Tidak ada tanggal yang sama muncul di dua partisi sekaligus (`validasi_urutan_waktu` menghentikan notebook jika urutan tanggal bocor).

In [4]:
def tentukan_skema_split(df: DataFrame) -> dict:
    """Hitung tanggal batas split kronologis ~70% train / 15% validation / 15% test."""
    harian = (
        df.groupBy(KOLOM_TANGGAL)
        .agg(F.count("*").alias("n"))
        .orderBy(KOLOM_TANGGAL)
        .collect()
    )
    if len(harian) < 3:
        raise ValueError("Butuh minimal 3 tanggal unik untuk split 70/15/15.")

    total = sum(int(r["n"]) for r in harian)
    target_train = total * RASIO_TRAIN
    target_valid = total * (RASIO_TRAIN + RASIO_VALID)

    kumulatif = 0
    tanggal_train_akhir = None
    tanggal_valid_akhir = None
    for row in harian:
        kumulatif += int(row["n"])
        tgl = row[KOLOM_TANGGAL]
        if tanggal_train_akhir is None and kumulatif >= target_train:
            tanggal_train_akhir = tgl
        if tanggal_valid_akhir is None and kumulatif >= target_valid:
            tanggal_valid_akhir = tgl
            break

    if tanggal_train_akhir is None or tanggal_valid_akhir is None:
        raise ValueError("Gagal menentukan batas tanggal split 70/15/15.")
    if not tanggal_train_akhir < tanggal_valid_akhir:
        raise ValueError("Tanggal batas train dan validation bertabrakan.")

    return {
        "mode": "kronologis_70_15_15",
        "rasio_target": {"train": RASIO_TRAIN, "validation": RASIO_VALID, "test": round(1.0 - RASIO_TRAIN - RASIO_VALID, 2)},
        "tanggal_train_akhir": str(tanggal_train_akhir),
        "tanggal_valid_akhir": str(tanggal_valid_akhir),
    }


def terapkan_split(df: DataFrame, skema: dict) -> DataFrame:
    """Beri label 'train', 'validation', atau 'test' sesuai batas tanggal 70/15/15."""
    tgl_train = F.lit(skema["tanggal_train_akhir"]).cast("date")
    tgl_valid = F.lit(skema["tanggal_valid_akhir"]).cast("date")
    return (
        df.withColumn(
            "split",
            F.when(F.col(KOLOM_TANGGAL) <= tgl_train, F.lit("train"))
            .when(F.col(KOLOM_TANGGAL) <= tgl_valid, F.lit("validation"))
            .otherwise(F.lit("test")),
        )
        .persist(StorageLevel.MEMORY_AND_DISK)
    )


def validasi_urutan_waktu(df_split: DataFrame) -> dict:
    """Pastikan train < validation < test secara tanggal (anti-leakage)."""
    rentang = {
        r["split"]: (r["min_tanggal"], r["max_tanggal"])
        for r in df_split.groupBy("split")
        .agg(
            F.min(KOLOM_TANGGAL).alias("min_tanggal"),
            F.max(KOLOM_TANGGAL).alias("max_tanggal"),
        )
        .collect()
    }
    if set(rentang) != {"train", "validation", "test"}:
        raise ValueError(f"Split tidak lengkap: {rentang}")

    if not rentang["train"][1] < rentang["validation"][0]:
        raise ValueError(
            "Bocor waktu: max(train) harus lebih awal dari min(validation). "
            f"Didapat: {rentang}"
        )
    if not rentang["validation"][1] < rentang["test"][0]:
        raise ValueError(
            "Bocor waktu: max(validation) harus lebih awal dari min(test). "
            f"Didapat: {rentang}"
        )
    return rentang


def siapkan_split(df: DataFrame, versi: str, skema: dict) -> DataFrame:
    """Terapkan split 70/15/15, validasi urutan waktu, dan cetak ringkasannya."""
    df_split = terapkan_split(df, skema)
    rentang = validasi_urutan_waktu(df_split)

    ringkasan = (
        df_split.groupBy("split")
        .agg(
            F.count("*").alias("baris"),
            F.countDistinct("titik_lokasi").alias("titik"),
            F.min(KOLOM_TANGGAL).alias("tanggal_min"),
            F.max(KOLOM_TANGGAL).alias("tanggal_max"),
        )
        .orderBy("tanggal_min")
    )
    print(f"\n=== Split {versi} ({skema['mode']}) ===")
    ringkasan.show(truncate=False)

    jumlah = {r["split"]: r["baris"] for r in ringkasan.collect()}
    total = sum(jumlah.values())
    for bagian in ("train", "validation", "test"):
        if jumlah.get(bagian, 0) == 0:
            raise ValueError(f"Bagian '{bagian}' kosong. Cek data dan skema split.")

    proporsi = {k: f"{v / total * 100:.2f}%" for k, v in jumlah.items()}
    print("Proporsi aktual :", proporsi)
    print(
        "Rentang tanggal :",
        {k: (str(v[0]), str(v[1])) for k, v in rentang.items()},
    )
    return df_split


def pecah_partisi_split(
    df_split: DataFrame,
) -> tuple[DataFrame, DataFrame, DataFrame, DataFrame]:
    """Ekstrak (train_tuning, valid, train_full, test) dari DataFrame split."""
    train_tuning = df_split.filter(F.col("split") == "train")
    valid = df_split.filter(F.col("split") == "validation")
    train_full = df_split.filter(F.col("split") != "test")
    test = df_split.filter(F.col("split") == "test")

    if LATIH_TANPA_OUTLIER and "is_outlier" in df_split.columns:
        train_tuning = train_tuning.filter(F.col("is_outlier") == 0)
        train_full = train_full.filter(F.col("is_outlier") == 0)

    return train_tuning, valid, train_full, test

## 3. Evaluasi & Baseline Naif

Baseline menggunakan rata-rata historis `volume_sampah_perhari` per `titik_lokasi` dari data train (`is_outlier == 0`). Karena volume sampah tidak pernah negatif, nilai prediksi pada evaluasi di-clamp minimal `0,0 m³`.

In [5]:
def metrik_regresi(df: DataFrame, pred_col: str = "prediction") -> dict[str, float]:
    """Hitung RMSE, MAE, dan R² (prediksi di-clamp >= 0 secara fisik)."""
    df_eval = df.withColumn(
        "_pred_nonneg", F.greatest(F.lit(0.0), F.col(pred_col))
    )
    hasil = {}
    for metrik in ("rmse", "mae", "r2"):
        evaluator = RegressionEvaluator(
            labelCol=TARGET, predictionCol="_pred_nonneg", metricName=metrik
        )
        hasil[metrik] = float(evaluator.evaluate(df_eval))
    return hasil


def hitung_baseline(
    train_df: DataFrame, eval_df: DataFrame
) -> tuple[DataFrame, dict[str, float]]:
    """Baseline naif: rata-rata historis per titik dari data train."""
    lookup = train_df.groupBy("titik_lokasi").agg(
        F.mean(TARGET).alias("prediction_baseline"),
        F.count("*").alias("n_historis"),
    )
    rata_global = float(train_df.agg(F.mean(TARGET)).first()[0])
    hasil = (
        eval_df.join(lookup, on="titik_lokasi", how="left")
        .withColumn(
            "prediction_baseline",
            F.coalesce(F.col("prediction_baseline"), F.lit(rata_global)),
        )
        .withColumn(
            "n_historis",
            F.coalesce(F.col("n_historis"), F.lit(0)),
        )
    )
    return hasil, metrik_regresi(hasil, "prediction_baseline")


def buat_baris_metrik(
    nama_model: str,
    versi_fitur: str,
    metrik_valid: dict[str, float],
    metrik_test: dict[str, float],
    durasi_latih_detik: float,
    n_valid_dinilai: int,
    n_valid_total: int,
    n_test_dinilai: int,
    n_test_total: int,
    catatan: str,
) -> dict:
    """Format satu baris ringkasan metrik model untuk tabel perbandingan."""
    return {
        "model": nama_model,
        "versi_fitur": versi_fitur,
        "rmse_valid": metrik_valid["rmse"],
        "mae_valid": metrik_valid["mae"],
        "r2_valid": metrik_valid["r2"],
        "rmse_test": metrik_test["rmse"],
        "mae_test": metrik_test["mae"],
        "r2_test": metrik_test["r2"],
        "durasi_latih_detik": durasi_latih_detik,
        "n_valid_dinilai": n_valid_dinilai,
        "n_valid_total": n_valid_total,
        "n_test_dinilai": n_test_dinilai,
        "n_test_total": n_test_total,
        "catatan": catatan,
    }

## 4. Linear Regression (Pembanding)

Pipeline: `StringIndexer -> OneHotEncoder -> VectorAssembler -> StandardScaler(withMean=False) -> LinearRegression(regParam=0.01, elasticNetParam=0.1)`. Regularisasi ringan mencegah matriks singular pada OHE `titik_lokasi`.

In [6]:
def buat_indexers(
    kolom_kategorik: list[str] = KOLOM_KATEGORIK,
) -> list[StringIndexer]:
    """Bangun tahap StringIndexer untuk seluruh kolom kategorik."""
    return [
        StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
        for c in kolom_kategorik
    ]


def bangun_pipeline_lr(versi: str) -> Pipeline:
    """Pipeline LR: StringIndexer -> OneHotEncoder -> StandardScaler -> LinearRegression."""
    numerik = fitur_numerik_untuk_versi(versi)
    indexers = buat_indexers()
    ohe = OneHotEncoder(
        inputCols=[f"{c}_idx" for c in KOLOM_KATEGORIK],
        outputCols=[f"{c}_ohe" for c in KOLOM_KATEGORIK],
        handleInvalid="keep",
    )
    assembler = VectorAssembler(
        inputCols=numerik + [f"{c}_ohe" for c in KOLOM_KATEGORIK],
        outputCol="features",
        handleInvalid="skip",
    )
    scaler = StandardScaler(
        inputCol="features",
        outputCol="features_scaled",
        withMean=False,
        withStd=True,
    )
    lr = LinearRegression(
        featuresCol="features_scaled",
        labelCol=TARGET,
        predictionCol="prediction",
        maxIter=100,
        regParam=0.01,
        elasticNetParam=0.1,
    )
    return Pipeline(stages=indexers + [ohe, assembler, scaler, lr])


def latih_evaluasi_lr(df_split: DataFrame, versi: str, skema: dict) -> dict:
    """Latih LR pada train, evaluasi di validation, refit pada train penuh, uji test."""
    train_tuning, valid, train_full, test = pecah_partisi_split(df_split)

    mulai = time.perf_counter()
    model_valid = bangun_pipeline_lr(versi).fit(train_tuning)
    durasi_valid = time.perf_counter() - mulai
    pred_valid = model_valid.transform(valid)
    metrik_valid = metrik_regresi(pred_valid)

    mulai = time.perf_counter()
    model_final = bangun_pipeline_lr(versi).fit(train_full)
    durasi_final = time.perf_counter() - mulai
    pred_test = model_final.transform(test)
    metrik_test = metrik_regresi(pred_test)

    numerik = fitur_numerik_untuk_versi(versi)
    return {
        "model_name": "LinearRegression",
        "versi_fitur": versi,
        "model": model_final,
        "params": {"regParam": 0.01, "elasticNetParam": 0.1, "maxIter": 100},
        "max_bins": None,
        "metrik_valid": metrik_valid,
        "metrik_test": metrik_test,
        "durasi_latih_detik": round(durasi_valid + durasi_final, 3),
        "n_valid_dinilai": pred_valid.count(),
        "n_valid_total": valid.count(),
        "n_test_dinilai": pred_test.count(),
        "n_test_total": test.count(),
        "fitur_numerik": numerik,
        "fitur_kategorik": list(KOLOM_KATEGORIK),
        "fitur_model": numerik + [f"{c}_ohe" for c in KOLOM_KATEGORIK],
        "data_train_full": train_full,
        "data_test": test,
        "skema_split": skema,
    }

## 5. Model Tree: Random Forest & Gradient-Boosted Trees (`GBTRegressor`)

Kedua model tree memakai `StringIndexer` (`maxBins >= kardinalitas terbesar + 2` dihitung terpisah untuk `train` dan `train_full`).

- **Random Forest:** `numTrees=[30, 50]`, `maxDepth=[6, 8]`, `minInstancesPerNode=[5, 10]`, `subsamplingRate=0.8`, `featureSubsetStrategy="onethird"`.
- **Gradient-Boosted Trees (`GBTRegressor`):** melatih pohon secara berurutan untuk memperbaiki residual dari pohon sebelumnya (`maxIter=[30, 50]`, `maxDepth=[4, 6]`, `stepSize=0.1`, `subsamplingRate=0.8`, `minInstancesPerNode=5`).

In [7]:
def kardinalitas_kategorik(df: DataFrame) -> dict[str, int]:
    """Hitung jumlah nilai unik tiap kolom kategorik."""
    return {
        kolom: int(df.select(kolom).distinct().count())
        for kolom in KOLOM_KATEGORIK
    }


def bangun_pipeline_rf(
    versi: str,
    num_trees: int,
    max_depth: int,
    min_instances: int,
    max_bins: int,
) -> Pipeline:
    """Pipeline RF memakai StringIndexer dan subsampling untuk stabilitas."""
    numerik = fitur_numerik_untuk_versi(versi)
    assembler = VectorAssembler(
        inputCols=numerik + [f"{c}_idx" for c in KOLOM_KATEGORIK],
        outputCol="features",
        handleInvalid="skip",
    )
    rf = RandomForestRegressor(
        featuresCol="features",
        labelCol=TARGET,
        predictionCol="prediction",
        numTrees=num_trees,
        maxDepth=max_depth,
        minInstancesPerNode=min_instances,
        maxBins=max_bins,
        subsamplingRate=0.8,
        featureSubsetStrategy="onethird",
        maxMemoryInMB=64,
        seed=SEED,
    )
    return Pipeline(stages=buat_indexers() + [assembler, rf])


def bangun_pipeline_gbt(
    versi: str,
    max_iter: int,
    max_depth: int,
    step_size: float,
    max_bins: int,
) -> Pipeline:
    """Pipeline Gradient-Boosted Trees (GBTRegressor) memakai StringIndexer."""
    numerik = fitur_numerik_untuk_versi(versi)
    assembler = VectorAssembler(
        inputCols=numerik + [f"{c}_idx" for c in KOLOM_KATEGORIK],
        outputCol="features",
        handleInvalid="skip",
    )
    gbt = GBTRegressor(
        featuresCol="features",
        labelCol=TARGET,
        predictionCol="prediction",
        maxIter=max_iter,
        maxDepth=max_depth,
        stepSize=step_size,
        minInstancesPerNode=5,
        subsamplingRate=0.8,
        maxBins=max_bins,
        maxMemoryInMB=64,
        seed=SEED,
    )
    return Pipeline(stages=buat_indexers() + [assembler, gbt])


def _jalankan_tuning_tree(
    df_split: DataFrame,
    versi: str,
    skema: dict,
    model_name: str,
    daftar_params: list[dict],
    builder_fn,
) -> dict:
    """Grid search generik di validation lalu refit pada train_full."""
    train_tuning, valid, train_full, test = pecah_partisi_split(df_split)

    kardinalitas = kardinalitas_kategorik(train_tuning)
    max_bins = max(MAX_BINS_MINIMUM, max(kardinalitas.values()) + 2)
    print(f"\n=== {model_name} {versi} ===")
    print(f"Kardinalitas train     : {kardinalitas} -> maxBins={max_bins}")

    n_valid = valid.count()
    catatan_tuning: list[dict] = []
    rmse_terbaik = float("inf")
    params_terbaik = None
    total_durasi = 0.0

    for params in daftar_params:
        pipeline = builder_fn(versi=versi, max_bins=max_bins, **params)
        mulai = time.perf_counter()
        model = pipeline.fit(train_tuning)
        durasi = time.perf_counter() - mulai
        total_durasi += durasi

        pred_valid = model.transform(valid)
        n_dinilai = pred_valid.count()
        metrik = metrik_regresi(pred_valid)
        baris = {
            "model": model_name,
            "versi_fitur": versi,
            **params,
            "max_bins": max_bins,
            "rmse_valid": metrik["rmse"],
            "mae_valid": metrik["mae"],
            "r2_valid": metrik["r2"],
            "durasi_latih_detik": round(durasi, 3),
            "n_valid_dinilai": n_dinilai,
            "n_valid_total": n_valid,
        }
        catatan_tuning.append(baris)
        ringkas_param = " ".join(f"{k}={v}" for k, v in params.items())
        print(
            f"  {ringkas_param} -> RMSE valid={metrik['rmse']:.4f}, "
            f"MAE={metrik['mae']:.4f}, R2={metrik['r2']:.4f} "
            f"({durasi:.1f} dtk, {n_dinilai}/{n_valid})"
        )
        if metrik["rmse"] < rmse_terbaik:
            rmse_terbaik = metrik["rmse"]
            params_terbaik = dict(params)

    if params_terbaik is None:
        raise RuntimeError(f"Tuning {model_name} gagal menghasilkan model.")

    kardinalitas_full = kardinalitas_kategorik(train_full)
    max_bins_final = max(MAX_BINS_MINIMUM, max(kardinalitas_full.values()) + 2)
    print(f"maxBins refit final    : {max_bins_final} (train_full: {kardinalitas_full})")

    pipeline_final = builder_fn(versi=versi, max_bins=max_bins_final, **params_terbaik)
    mulai = time.perf_counter()
    model_final = pipeline_final.fit(train_full)
    durasi_final = time.perf_counter() - mulai

    pred_test = model_final.transform(test)
    n_test_dinilai = pred_test.count()
    metrik_test = metrik_regresi(pred_test)

    metrik_valid_terbaik = min(catatan_tuning, key=lambda r: r["rmse_valid"])
    df_tuning = pd.DataFrame(catatan_tuning).sort_values("rmse_valid")
    numerik = fitur_numerik_untuk_versi(versi)

    print(
        f"Total tuning {model_name} {versi}: {total_durasi:.1f} dtk | "
        f"refit: {durasi_final:.1f} dtk"
    )

    return {
        "model_name": model_name,
        "versi_fitur": versi,
        "model": model_final,
        "params": params_terbaik,
        "max_bins": max_bins_final,
        "metrik_valid": {
            "rmse": metrik_valid_terbaik["rmse_valid"],
            "mae": metrik_valid_terbaik["mae_valid"],
            "r2": metrik_valid_terbaik["r2_valid"],
        },
        "metrik_test": metrik_test,
        "durasi_latih_detik": round(durasi_final, 3),
        "durasi_tuning_total_detik": round(total_durasi, 3),
        "n_valid_dinilai": metrik_valid_terbaik["n_valid_dinilai"],
        "n_valid_total": n_valid,
        "n_test_dinilai": n_test_dinilai,
        "n_test_total": test.count(),
        "tabel_tuning": df_tuning,
        "fitur_numerik": numerik,
        "fitur_kategorik": list(KOLOM_KATEGORIK),
        "fitur_model": numerik + [f"{c}_idx" for c in KOLOM_KATEGORIK],
        "data_train_full": train_full,
        "data_test": test,
        "skema_split": skema,
    }


def tuning_rf(df_split: DataFrame, versi: str, skema: dict) -> dict:
    """Grid search RandomForestRegressor."""
    kombinasi = [
        {"num_trees": nt, "max_depth": md, "min_instances": mi}
        for nt in GRID_RF["numTrees"]
        for md in GRID_RF["maxDepth"]
        for mi in GRID_RF["minInstancesPerNode"]
    ]
    return _jalankan_tuning_tree(
        df_split, versi, skema, "RandomForest", kombinasi, bangun_pipeline_rf
    )


def tuning_gbt(df_split: DataFrame, versi: str, skema: dict) -> dict:
    """Grid search GBTRegressor."""
    kombinasi = [
        {"max_iter": mi, "max_depth": md, "step_size": ss}
        for mi in GRID_GBT["maxIter"]
        for md in GRID_GBT["maxDepth"]
        for ss in GRID_GBT["stepSize"]
    ]
    return _jalankan_tuning_tree(
        df_split, versi, skema, "GBT", kombinasi, bangun_pipeline_gbt
    )

## 6. Eksperimen: Baseline, LR, RF, dan GBT (`v1` & `v2`)

Mengevaluasi 7 konfigurasi pada split kronologis **70 / 15 / 15**: **Baseline**, **LR v1**, **RF v1**, **GBT v1**, **LR v2**, **RF v2**, dan **GBT v2**.

### 6.1 Jalankan eksperimen & susun tabel metrik

In [8]:
df_v1 = baca_parquet_fitur(FEATURES_V1, "v1")
df_v2 = baca_parquet_fitur(FEATURES_V2, "v2")

skema = tentukan_skema_split(df_v1)
print("Skema split:", json.dumps(skema, indent=2))

hasil_metrik: list[dict] = []
model_tersimpan: dict[tuple[str, str], dict] = {}
tabel_tuning: dict[tuple[str, str], pd.DataFrame] = {}

for versi, df_fitur in (("v1", df_v1), ("v2", df_v2)):
    print(f"\n{'=' * 70}\nMEMPROSES FITUR {versi.upper()}\n{'=' * 70}")
    split = siapkan_split(df_fitur, versi, skema)

    if versi == "v1":
        train_tuning_b, valid_b, train_full_b, test_b = pecah_partisi_split(split)
        _, baseline_valid = hitung_baseline(train_tuning_b, valid_b)
        _, baseline_test = hitung_baseline(train_full_b, test_b)
        n_valid_b = valid_b.count()
        n_test_b = test_b.count()
        hasil_metrik.append(
            buat_baris_metrik(
                nama_model="baseline (mean per titik)",
                versi_fitur="-",
                metrik_valid=baseline_valid,
                metrik_test=baseline_test,
                durasi_latih_detik=0.0,
                n_valid_dinilai=n_valid_b,
                n_valid_total=n_valid_b,
                n_test_dinilai=n_test_b,
                n_test_total=n_test_b,
                catatan="baseline naif (train tanpa outlier ekstrem)",
            )
        )

    for nama_fungsi, runner, cat_fn in (
        ("LinearRegression", latih_evaluasi_lr, lambda info: "OHE + StandardScaler + L2; refit train penuh"),
        ("RandomForest", tuning_rf, lambda info: f"params={info['params']}, maxBins={info['max_bins']}"),
        ("GBT", tuning_gbt, lambda info: f"params={info['params']}, maxBins={info['max_bins']}"),
    ):
        info = runner(split, versi, skema)
        model_tersimpan[(versi, nama_fungsi)] = info
        if "tabel_tuning" in info:
            tabel_tuning[(nama_fungsi, versi)] = info["tabel_tuning"]
        hasil_metrik.append(
            buat_baris_metrik(
                nama_model=nama_fungsi,
                versi_fitur=versi,
                metrik_valid=info["metrik_valid"],
                metrik_test=info["metrik_test"],
                durasi_latih_detik=info["durasi_latih_detik"],
                n_valid_dinilai=info["n_valid_dinilai"],
                n_valid_total=info["n_valid_total"],
                n_test_dinilai=info["n_test_dinilai"],
                n_test_total=info["n_test_total"],
                catatan=cat_fn(info),
            )
        )
        print(
            f"{nama_fungsi} {versi}: RMSE valid={info['metrik_valid']['rmse']:.4f}, "
            f"RMSE test={info['metrik_test']['rmse']:.4f}, R2 test={info['metrik_test']['r2']:.4f}"
        )

    split.unpersist()

kolom_metrik = [
    "model",
    "versi_fitur",
    "rmse_valid",
    "mae_valid",
    "r2_valid",
    "rmse_test",
    "mae_test",
    "r2_test",
    "durasi_latih_detik",
    "n_valid_dinilai",
    "n_valid_total",
    "n_test_dinilai",
    "n_test_total",
    "catatan",
]
df_metrik = pd.DataFrame(hasil_metrik)[kolom_metrik]
display(df_metrik)

Skema split: {
  "mode": "kronologis_70_15_15",
  "rasio_target": {
    "train": 0.7,
    "validation": 0.15,
    "test": 0.15
  },
  "tanggal_train_akhir": "2021-07-03",
  "tanggal_valid_akhir": "2021-10-02"
}

MEMPROSES FITUR V1



=== Split v1 (kronologis_70_15_15) ===


+----------+------+-----+-----------+-----------+
|split     |baris |titik|tanggal_min|tanggal_max|
+----------+------+-----+-----------+-----------+
|train     |748979|2987 |2018-01-01 |2021-07-03 |
|validation|160615|1768 |2021-07-04 |2021-10-02 |
|test      |158968|1767 |2021-10-03 |2021-12-31 |
+----------+------+-----+-----------+-----------+



Proporsi aktual : {'train': '70.09%', 'validation': '15.03%', 'test': '14.88%'}
Rentang tanggal : {'validation': ('2021-07-04', '2021-10-02'), 'test': ('2021-10-03', '2021-12-31'), 'train': ('2018-01-01', '2021-07-03')}


LinearRegression v1: RMSE valid=4.0781, RMSE test=4.2482, R2 test=-0.0897



=== RandomForest v1 ===
Kardinalitas train     : {'titik_lokasi': 2987, 'kecamatan': 45, 'wilayah': 5} -> maxBins=2989


... (36 outputs left out)

  max_iter=50 max_depth=6 step_size=0.1 -> RMSE valid=4.7157, MAE=2.0758, R2=-0.8723 (245.3 dtk, 160593/160615)


maxBins refit final    : 2992 (train_full: {'titik_lokasi': 2990, 'kecamatan': 45, 'wilayah': 5})


Total tuning GBT v2: 629.9 dtk | refit: 68.2 dtk
GBT v2: RMSE valid=3.5662, RMSE test=3.7416, R2 test=0.1549


,model,versi_fitur,rmse_valid,mae_valid,r2_valid,rmse_test,mae_test,r2_test,durasi_latih_detik,n_valid_dinilai,n_valid_total,n_test_dinilai,n_test_total,catatan
0,baseline (mean per titik),-,3.942634,1.897103,-0.308862,4.114131,1.513561,-0.022016,0.000,160615,160615,158968,158968,baseline naif (train tanpa outlier ekstrem)
1,LinearRegression,v1,4.078116,2.167474,-0.400362,4.248211,1.808973,-0.089717,43.888,160615,160615,158968,158968,OHE + StandardScaler + L2; refit train penuh
2,RandomForest,v1,3.928308,1.923460,-0.299368,4.099311,1.479452,-0.014666,126.352,160615,160615,158968,158968,"params={'num_trees': 50, 'max_depth': 8, 'min_..."
3,GBT,v1,4.116870,1.907744,-0.427103,4.925840,1.840622,-0.465083,130.313,160615,160615,158968,158968,"params={'max_iter': 30, 'max_depth': 6, 'step_..."
4,LinearRegression,v2,3.296981,0.947437,0.084836,3.740379,0.928404,0.155498,39.273,160593,160615,158918,158968,OHE + StandardScaler + L2; refit train penuh
5,RandomForest,v2,3.638965,1.416943,-0.114864,3.751976,0.746201,0.150253,91.948,160593,160615,158918,158968,"params={'num_trees': 50, 'max_depth': 6, 'min_..."
6,GBT,v2,3.566236,1.076315,-0.070746,3.741599,0.754642,0.154947,68.235,160593,160615,158918,158968,"params={'max_iter': 30, 'max_depth': 4, 'step_..."


### 6.2 Detail tuning Random Forest & GBT

Menampilkan seluruh kombinasi grid untuk `RF` dan `GBT` pada `v1` dan `v2`.

In [9]:
for (nama_model, versi), tabel in tabel_tuning.items():
    print(f"\nTuning {nama_model} {versi} (urut RMSE validation):")
    display(tabel)


Tuning RandomForest v1 (urut RMSE validation):


,model,versi_fitur,num_trees,max_depth,min_instances,max_bins,rmse_valid,mae_valid,r2_valid,durasi_latih_detik,n_valid_dinilai,n_valid_total
6,RandomForest,v1,50,8,5,2989,3.928308,1.923460,-0.299368,100.383,160615,160615
4,RandomForest,v1,50,6,5,2989,3.945065,1.953197,-0.310477,43.345,160615,160615
7,RandomForest,v1,50,8,10,2989,3.976371,1.970921,-0.331358,107.448,160615,160615
1,RandomForest,v1,30,6,10,2989,4.002924,2.019274,-0.349198,29.835,160615,160615
2,RandomForest,v1,30,8,5,2989,4.012678,1.988596,-0.355782,66.772,160615,160615
0,RandomForest,v1,30,6,5,2989,4.016152,2.038554,-0.358130,30.356,160615,160615
5,RandomForest,v1,50,6,10,2989,4.018496,2.029272,-0.359716,39.834,160615,160615
3,RandomForest,v1,30,8,10,2989,4.037749,1.990122,-0.372776,65.012,160615,160615



Tuning GBT v1 (urut RMSE validation):


,model,versi_fitur,max_iter,max_depth,step_size,max_bins,rmse_valid,mae_valid,r2_valid,durasi_latih_detik,n_valid_dinilai,n_valid_total
1,GBT,v1,30,6,0.1,2989,4.116870,1.907744,-0.427103,116.870,160615,160615
0,GBT,v1,30,4,0.1,2989,4.143900,2.016755,-0.445905,66.124,160615,160615
2,GBT,v1,50,4,0.1,2989,4.158669,2.016969,-0.456229,120.483,160615,160615
3,GBT,v1,50,6,0.1,2989,4.212173,1.922704,-0.493941,259.366,160615,160615



Tuning RandomForest v2 (urut RMSE validation):


,model,versi_fitur,num_trees,max_depth,min_instances,max_bins,rmse_valid,mae_valid,r2_valid,durasi_latih_detik,n_valid_dinilai,n_valid_total
5,RandomForest,v2,50,6,10,2989,3.638965,1.416943,-0.114864,70.181,160593,160615
4,RandomForest,v2,50,6,5,2989,3.693207,1.461394,-0.148348,66.681,160593,160615
1,RandomForest,v2,30,6,10,2989,3.720794,1.502491,-0.165568,46.620,160593,160615
0,RandomForest,v2,30,6,5,2989,3.922224,1.677803,-0.295183,41.915,160593,160615
3,RandomForest,v2,30,8,10,2989,4.186411,1.998008,-0.475537,80.480,160593,160615
7,RandomForest,v2,50,8,10,2989,4.320563,2.073137,-0.571618,139.256,160593,160615
2,RandomForest,v2,30,8,5,2989,4.646027,2.297071,-0.817313,87.442,160593,160615
6,RandomForest,v2,50,8,5,2989,5.163139,2.674772,-1.244367,146.957,160593,160615



Tuning GBT v2 (urut RMSE validation):


,model,versi_fitur,max_iter,max_depth,step_size,max_bins,rmse_valid,mae_valid,r2_valid,durasi_latih_detik,n_valid_dinilai,n_valid_total
0,GBT,v2,30,4,0.1,2989,3.566236,1.076315,-0.070746,78.583,160593,160615
2,GBT,v2,50,4,0.1,2989,3.581747,1.037930,-0.080081,146.211,160593,160615
1,GBT,v2,30,6,0.1,2989,4.355076,1.883952,-0.596827,159.783,160593,160615
3,GBT,v2,50,6,0.1,2989,4.715732,2.075767,-0.872253,245.297,160593,160615


## 7. Pemilihan Model Terbaik & Penyimpanan Artefak

Pemilihan model mengacu pada **RMSE validation** terkecil (bukan `test`) agar `test` tetap murni sebagai evaluasi akhir.

In [10]:
kandidat = (
    df_metrik[df_metrik["model"] != "baseline (mean per titik)"]
    .sort_values("rmse_valid")
)
baris_terbaik = kandidat.iloc[0]
kunci = (str(baris_terbaik["versi_fitur"]), str(baris_terbaik["model"]))
info_terbaik = model_tersimpan[kunci]
model_terbaik = info_terbaik["model"]

baseline_rmse_valid = float(
    df_metrik.loc[
        df_metrik["model"] == "baseline (mean per titik)", "rmse_valid"
    ].iloc[0]
)
if baris_terbaik["rmse_valid"] >= baseline_rmse_valid:
    print(
        "PERINGATAN: model terbaik belum mengalahkan baseline pada validation. "
        "Laporkan hasil apa adanya."
    )

VERSI_MODEL = (
    f"{info_terbaik['model_name'].lower()}_{info_terbaik['versi_fitur']}_"
    f"{datetime.now():%Y%m%d_%H%M%S}"
)
MODELS_DIR.mkdir(parents=True, exist_ok=True)
PATH_MODEL = MODELS_DIR / VERSI_MODEL

model_terbaik.write().overwrite().save(path_ke_uri(PATH_MODEL))

metadata = {
    "versi_model": VERSI_MODEL,
    "model": info_terbaik["model_name"],
    "versi_fitur": info_terbaik["versi_fitur"],
    "fitur_numerik": info_terbaik["fitur_numerik"],
    "fitur_kategorik": info_terbaik["fitur_kategorik"],
    "fitur_model": info_terbaik["fitur_model"],
    "parameter": {
        k: (int(v) if isinstance(v, int) else float(v) if isinstance(v, float) else v)
        for k, v in info_terbaik["params"].items()
    },
    "max_bins": info_terbaik["max_bins"],
    "skema_split": info_terbaik["skema_split"],
    "latih_tanpa_outlier": LATIH_TANPA_OUTLIER,
    "metrik_validation": info_terbaik["metrik_valid"],
    "metrik_test": info_terbaik["metrik_test"],
    "dilatih_pada": datetime.now().isoformat(timespec="seconds"),
    "catatan": "Split kronologis 70/15/15; prediksi dievaluasi dengan clamp >= 0.",
}
PATH_METADATA = MODELS_DIR / f"{VERSI_MODEL}_metadata.json"
PATH_METADATA.write_text(
    json.dumps(metadata, indent=2, default=str), encoding="utf-8"
)

PATH_METRIK = MODELS_DIR / f"{VERSI_MODEL}_metrik.csv"
df_metrik.to_csv(PATH_METRIK, index=False)

print(f"Model terbaik : {info_terbaik['model_name']} {info_terbaik['versi_fitur']}")
print(f"Parameter     : {info_terbaik['params']}")
print(f"RMSE valid    : {info_terbaik['metrik_valid']['rmse']:.4f}")
print(f"RMSE test     : {info_terbaik['metrik_test']['rmse']:.4f}")
print(f"R² test       : {info_terbaik['metrik_test']['r2']:.4f}")
print(f"Model         : {PATH_MODEL}")
print(f"Metadata      : {PATH_METADATA}")
print(f"Metrik        : {PATH_METRIK}")

Model terbaik : LinearRegression v2
Parameter     : {'regParam': 0.01, 'elasticNetParam': 0.1, 'maxIter': 100}
RMSE valid    : 3.2970
RMSE test     : 3.7404
R² test       : 0.1555
Model         : C:\Users\anand\OneDrive\Documents\Project\Sipeka\models\linearregression_v2_20261121_170617
Metadata      : C:\Users\anand\OneDrive\Documents\Project\Sipeka\models\linearregression_v2_20261121_170617_metadata.json
Metrik        : C:\Users\anand\OneDrive\Documents\Project\Sipeka\models\linearregression_v2_20261121_170617_metrik.csv


## 8. Analisis Error & Feature Importance (Test Set 15%)

Evaluasi error dihitung penuh di Spark (prediksi di-clamp `>= 0`), lalu hasil agregat dan sampel kecil divisualisasikan dengan Plotly.

> Definisi: `bias = prediction - actual` (positif = **overprediksi**, negatif = **underprediksi**).

### 8.1 Error per wilayah, per bulan, dan 10 titik MAE tertinggi

In [11]:
def ringkas_error(
    df_pred: DataFrame,
    group_cols: list[str],
    order_expr,
    limit: int | None = None,
) -> DataFrame:
    """Hitung jumlah baris, bias, MAE, dan RMSE error per grup."""
    hasil = (
        df_pred.groupBy(*group_cols)
        .agg(
            F.count("*").alias("n"),
            F.mean("error").alias("bias"),
            F.mean("abs_error").alias("mae"),
            F.sqrt(F.mean(F.pow(F.col("error"), F.lit(2)))).alias("rmse"),
        )
        .orderBy(order_expr)
    )
    return hasil.limit(limit) if limit is not None else hasil


data_test = info_terbaik["data_test"]
pred_test = (
    model_terbaik.transform(data_test)
    .withColumn("prediction", F.greatest(F.lit(0.0), F.col("prediction")))
    .withColumn("error", F.col("prediction") - F.col(TARGET))
    .withColumn("abs_error", F.abs(F.col("error")))
)

n_test_total = data_test.count()
n_test_dinilai = pred_test.count()
print(f"Test dinilai: {n_test_dinilai}/{n_test_total} baris")

error_wilayah = ringkas_error(pred_test, ["wilayah"], F.desc("mae"))
error_bulan = ringkas_error(pred_test, ["bulan"], F.asc("bulan"))
titik_terburuk = ringkas_error(
    pred_test, ["titik_lokasi", "wilayah"], F.desc("mae"), limit=10
)

print("\nError per wilayah:")
error_wilayah.show(truncate=False)
print("\nError per bulan:")
error_bulan.show(truncate=False)
print("\n10 titik dengan MAE tertinggi:")
titik_terburuk.show(truncate=False)

Test dinilai: 158918/158968 baris



Error per wilayah:


+---------------+-----+-------------------+------------------+------------------+
|wilayah        |n    |bias               |mae               |rmse              |
+---------------+-----+-------------------+------------------+------------------+
|JAKARTA PUSAT  |14082|0.3834318221952485 |1.1852398015594896|11.57873744113767 |
|JAKARTA BARAT  |48400|0.5479113442573291 |0.9564123280298129|1.8362822365916862|
|JAKARTA TIMUR  |35146|0.33082489663429976|0.9461140634169362|1.2220118985376673|
|JAKARTA SELATAN|23580|0.5916849389389458 |0.9269010208042485|1.1432526929415776|
|JAKARTA UTARA  |37710|0.3122411484898256 |0.7809816597320505|1.535344742387314 |
+---------------+-----+-------------------+------------------+------------------+


Error per bulan:


+-----+-----+------------------+------------------+------------------+
|bulan|n    |bias              |mae               |rmse              |
+-----+-----+------------------+------------------+------------------+
|10.0 |51176|0.1785504481153224|0.7999108934485478|5.85639095266612  |
|11.0 |52979|0.4489979746294859|0.9339921676615235|2.425930109931567 |
|12.0 |54763|0.663717308007301 |1.04307583515243  |1.6896512621201891|
+-----+-----+------------------+------------------+------------------+


10 titik dengan MAE tertinggi:


+-------------------------------------------------------------------------+-------------+---+--------------------+------------------+------------------+
|titik_lokasi                                                             |wilayah      |n  |bias                |mae               |rmse              |
+-------------------------------------------------------------------------+-------------+---+--------------------+------------------+------------------+
|KALI / SUNGAI CILIWUNG EMPLASEMENT PA MANGGARAI/RATU HARHARI KUNINGAN    |JAKARTA PUSAT|90 |-34.19672933760274  |43.38879218971254 |143.84908048719305|
|BANJIR KANAL BARAT SEASONS CITY (JLN. INSPEKSI KALI ANYAR-JLN. PTB ANGKE)|JAKARTA BARAT|90 |-5.039318028837284  |14.218681029814123|23.974418181173572|
|WADUK PLUIT                                                              |JAKARTA UTARA|90 |-0.865954579739039  |9.731147677059031 |13.927979305498813|
|SEKATAN HDPE KALI SODETAN SEKRETARIS (JLN. TANJUNG DUREN UTARA)          |JAKARTA

### 8.2 Visualisasi sampel & ekspor figure

In [12]:
def simpan_figure(fig, nama_dasar: str) -> None:
    """Simpan figure Plotly ke HTML dan JSON di FIGURES_DIR lalu tampilkan."""
    fig.write_html(str(FIGURES_DIR / f"{nama_dasar}.html"))
    fig.write_json(str(FIGURES_DIR / f"{nama_dasar}.json"))
    fig.show()


FIGURES_DIR.mkdir(parents=True, exist_ok=True)

fraksi = min(1.0, SAMPEL_VISUALISASI / max(n_test_dinilai, 1))
sampel = (
    pred_test.select(TARGET, "prediction")
    .sample(False, fraksi, seed=SEED)
    .limit(SAMPEL_VISUALISASI)
    .toPandas()
)

if not sampel.empty:
    batas = float(max(sampel[TARGET].max(), sampel["prediction"].max()))
    fig_scatter = px.scatter(
        sampel,
        x=TARGET,
        y="prediction",
        opacity=0.35,
        title=(
            "Actual vs Predicted — "
            f"{info_terbaik['model_name']} {info_terbaik['versi_fitur']}"
        ),
        labels={
            TARGET: "Volume aktual (m³)",
            "prediction": "Volume prediksi (m³)",
        },
    )
    fig_scatter.add_shape(
        type="line",
        x0=0,
        y0=0,
        x1=batas,
        y1=batas,
        line=dict(color="red", dash="dash"),
    )
    simpan_figure(fig_scatter, "step6_actual_vs_predicted")

fig_wilayah = px.bar(
    error_wilayah.toPandas(),
    x="wilayah",
    y="mae",
    title="MAE per wilayah — data test (15%)",
    labels={"mae": "MAE (m³)", "wilayah": "Wilayah"},
)
simpan_figure(fig_wilayah, "step6_mae_per_wilayah")

fig_bulan = px.line(
    error_bulan.toPandas(),
    x="bulan",
    y="bias",
    markers=True,
    title="Bias prediksi per bulan — data test (15%)",
    labels={"bias": "Mean error (m³)", "bulan": "Bulan"},
)
simpan_figure(fig_bulan, "step6_bias_per_bulan")

### 8.3 Feature importance model tree

In [13]:
if info_terbaik["model_name"] in {"RandomForest", "GBT"}:
    tree_model = model_terbaik.stages[-1]
    bobot = tree_model.featureImportances.toArray()
    df_importance = (
        pd.DataFrame({"fitur": info_terbaik["fitur_model"], "importance": bobot})
        .sort_values("importance", ascending=False)
        .reset_index(drop=True)
    )
    PATH_IMPORTANCE = MODELS_DIR / f"{VERSI_MODEL}_feature_importance.csv"
    df_importance.to_csv(PATH_IMPORTANCE, index=False)
    print(f"Feature importance {info_terbaik['model_name']} (top 15):")
    display(df_importance.head(15))
    print(f"Disimpan: {PATH_IMPORTANCE}")
else:
    print(
        "Model terbaik bukan model tree; feature importance tidak dihitung."
    )

Model terbaik bukan model tree; feature importance tidak dihitung.


## 9. Prediksi Masa Depan (`PrediktorMasaDepan`)

Menyediakan antarmuka `prediksi(tanggal, titik_lokasi)` sebagai jembatan menuju `src/modeling/score.py` dan dashboard.

### 9.1 Helper fitur kalender & kelas `PrediktorMasaDepan`

In [14]:
DAFTAR_HARI_LIBUR: set[str] = set()


def tambah_fitur_kalender(
    df: DataFrame, kolom_tanggal: str = KOLOM_TANGGAL
) -> DataFrame:
    """Bangun seluruh fitur kalender dengan definisi identik Step 4."""
    pi = math.pi
    bulan = F.month(kolom_tanggal)
    dow = F.dayofweek(kolom_tanggal)

    if DAFTAR_HARI_LIBUR:
        ekspresi_libur = F.when(
            F.date_format(kolom_tanggal, "yyyy-MM-dd").isin(
                *sorted(DAFTAR_HARI_LIBUR)
            ),
            1,
        ).otherwise(0)
    else:
        ekspresi_libur = F.lit(0)

    return df.withColumns(
        {
            "tahun": F.year(kolom_tanggal),
            "bulan": bulan,
            "hari_dalam_bulan": F.dayofmonth(kolom_tanggal),
            "hari_dalam_minggu": dow,
            "is_weekend": F.when(dow.isin(1, 7), 1).otherwise(0),
            "minggu_ke": F.ceil(F.dayofmonth(kolom_tanggal) / 7).cast("int"),
            "kuartal": F.quarter(kolom_tanggal),
            "hari_ke_dalam_tahun": F.dayofyear(kolom_tanggal),
            "bulan_sin": F.sin(2 * pi * (bulan - 1) / 12),
            "bulan_cos": F.cos(2 * pi * (bulan - 1) / 12),
            "hari_sin": F.sin(2 * pi * (dow - 1) / 7),
            "hari_cos": F.cos(2 * pi * (dow - 1) / 7),
            "is_musim_hujan": F.when(
                bulan.isin(list(BULAN_MUSIM_HUJAN)), 1
            ).otherwise(0),
            "is_hari_libur": ekspresi_libur,
        }
    )


class PrediktorMasaDepan:
    """Wrapper PipelineModel untuk prediksi satu tanggal dan satu titik_lokasi."""

    def __init__(
        self,
        path_model: str | Path,
        df_historis: DataFrame,
        metadata: dict,
    ) -> None:
        uri_model = path_ke_uri(path_model)
        try:
            self.model = PipelineModel.load(uri_model)
        except Exception:
            self.model = PipelineModel._from_java(
                SPARK._jvm.org.apache.spark.ml.PipelineModel.load(uri_model)
            )
        self.metadata = metadata
        self.versi_fitur = metadata["versi_fitur"]
        self.fitur_numerik = list(metadata["fitur_numerik"])

        df_ref = (
            df_historis.filter(F.col("is_outlier") == 0)
            if "is_outlier" in df_historis.columns
            else df_historis
        )

        jendela = Window.partitionBy("titik_lokasi").orderBy(
            F.desc("jumlah"), F.asc("kecamatan"), F.asc("wilayah")
        )
        self.lookup_lokasi = (
            df_ref.groupBy("titik_lokasi", "kecamatan", "wilayah")
            .agg(F.count("*").alias("jumlah"))
            .withColumn("peringkat", F.row_number().over(jendela))
            .filter(F.col("peringkat") == 1)
            .select("titik_lokasi", "kecamatan", "wilayah")
            .cache()
        )

        kolom_historis = [c for c in FITUR_HISTORIS if c in self.fitur_numerik]
        if kolom_historis:
            hist_per_titik = df_ref.groupBy("titik_lokasi").agg(
                *[F.mean(c).alias(c) for c in kolom_historis]
            )
            rata_global = (
                df_ref.agg(*[F.mean(c).alias(c) for c in kolom_historis])
                .first()
                .asDict()
            )
            rata_global = {
                k: (v if v is not None else 0.0) for k, v in rata_global.items()
            }
            self.hist_default = hist_per_titik.fillna(rata_global).cache()
        else:
            self.hist_default = None

    def bersihkan_cache(self) -> None:
        """Lepas DataFrame cache internal jika prediktor tidak lagi dipakai."""
        self.lookup_lokasi.unpersist()
        if self.hist_default is not None:
            self.hist_default.unpersist()

    def prediksi(self, tanggal: str, titik_lokasi: str) -> float:
        """Prediksi volume (m³) untuk tanggal 'YYYY-MM-DD' dan satu titik (>= 0)."""
        data = SPARK.range(1).select(
            F.lit(tanggal).cast("string").alias("tanggal_str"),
            F.lit(titik_lokasi).cast("string").alias("titik_lokasi"),
        )
        data = data.withColumn(
            KOLOM_TANGGAL, F.to_date("tanggal_str", "yyyy-MM-dd")
        ).drop("tanggal_str")

        if data.filter(F.col(KOLOM_TANGGAL).isNull()).count() > 0:
            raise ValueError("Format tanggal harus 'YYYY-MM-DD'.")

        if (
            self.lookup_lokasi.filter(F.col("titik_lokasi") == titik_lokasi).count()
            == 0
        ):
            raise ValueError(
                f"titik_lokasi '{titik_lokasi}' tidak ada di data historis."
            )

        data = tambah_fitur_kalender(data)
        data = data.join(self.lookup_lokasi, on="titik_lokasi", how="left")

        if self.versi_fitur == "v2":
            if self.hist_default is None:
                raise ValueError("Model v2 butuh fitur historis yang tidak tersedia.")
            data = data.join(self.hist_default, on="titik_lokasi", how="left")

        hilang = [c for c in self.fitur_numerik if c not in data.columns]
        if hilang:
            raise ValueError(f"Fitur prediksi tidak lengkap: {hilang}")

        baris = self.model.transform(data).select("prediction").first()
        if baris is None or baris["prediction"] is None:
            raise RuntimeError("Model tidak menghasilkan prediksi.")
        return max(0.0, float(baris["prediction"]))

### 9.2 Uji cepat prediksi satu hari setelah data train

In [15]:
prediktor = PrediktorMasaDepan(
    path_model=PATH_MODEL,
    df_historis=info_terbaik["data_train_full"],
    metadata=metadata,
)

tanggal_akhir_train = (
    info_terbaik["data_train_full"].agg(F.max(KOLOM_TANGGAL)).first()[0]
)
contoh_tanggal = (tanggal_akhir_train + timedelta(days=1)).isoformat()
contoh_titik = info_terbaik["data_train_full"].select("titik_lokasi").first()[0]
contoh_prediksi = prediktor.prediksi(contoh_tanggal, contoh_titik)

print(
    f"Contoh prediksi {contoh_titik} pada {contoh_tanggal}: "
    f"{contoh_prediksi:.2f} m³"
)

Contoh prediksi ALIRAN KALI BARU BARAT IISIP LENTENG AGUNG pada 2021-10-03: 1.67 m³


## 10. Interpretasi Hasil

Seluruh angka di bawah diambil langsung dari `df_metrik` dan analisis error Bagian 8 pada split kronologis **70 / 15 / 15** (`LATIH_TANPA_OUTLIER = True`, prediksi di-clamp `>= 0`).

### 10.1 Perbandingan model

| Model | Fitur | RMSE valid | MAE valid | R² valid | RMSE test | MAE test | R² test |
|-------|-------|-----------:|----------:|---------:|----------:|---------:|--------:|
| Baseline (mean/titik) | `-` | 3,9426 | 1,8971 | -0,3089 | 4,1141 | 1,5136 | -0,0220 |
| Linear Regression | `v1` | 4,0781 | 2,1675 | -0,4004 | 4,2482 | 1,8090 | -0,0897 |
| Random Forest | `v1` | 3,9283 | 1,9235 | -0,2994 | 4,0993 | 1,4795 | -0,0147 |
| GBT | `v1` | 4,1169 | 1,9077 | -0,4271 | 4,9258 | 1,8406 | -0,4651 |
| **Linear Regression** *(terpilih)* | **`v2`** | **3,2970** | **0,9474** | **0,0848** | **3,7404** | **0,9284** | **0,1555** |
| Random Forest | `v2` | 3,6390 | 1,4169 | -0,1149 | 3,7520 | 0,7462 | 0,1503 |
| GBT | `v2` | 3,5662 | 1,0763 | -0,0707 | 3,7416 | 0,7546 | 0,1549 |

### 10.2 Analisis peningkatan performa

- **Dampak split 70 / 15 / 15 (`train` 70,09%, `validation` 15,03%, `test` 14,88%):** data latih mencakup `2018-01-01` s.d. `2021-07-03` (`748.979` baris) sehingga model mempelajari pola operasional 2021 sebelum diuji pada `2021-07-04`–`2021-10-02` (`160.615` baris) dan `2021-10-03`–`2021-12-31` (`158.968` baris).
- **Dampak fitur `v2` yang diperkaya + normalisasi nama titik:** seluruh model `v2` (LR, GBT, RF) mengalahkan baseline secara konsisten pada `validation` maupun `test`, dengan **`R² test` positif (~`0,150`–`0,156`)** dan **`MAE test` turun lebih dari separuh** dari `1,51 m³` (baseline) menjadi `0,75 m³` (RF/GBT `v2`) dan `0,93 m³` (LR `v2`).
- **Mengapa `Linear Regression v2` menang pada RMSE validation (`3,2970`) & RMSE test (`3,7404`)?** Kombinasi fitur historis kontinu bebas-leakage (`titik_hist_mean`, `titik_hist_zero_rate`, `rolling_mean_7/14/30`, `rolling_std_7`, `rolling_max_7`, `lag_1/7/14`) bersama One-Hot Encoding lokasi + regularisasi ElasticNet (`regParam=0,01`, `elasticNetParam=0,1`) membuat hubungan linear antar fitur historis dan target sangat efektif tanpa batasan kedalaman tree. Sementara **RF `v2` (`MAE = 0,7462 m³`)** dan **GBT `v2` (`MAE = 0,7546 m³`)** unggul dalam ketepatan median/absolut harian karena tidak terpengaruh sedikit lonjakan ekstrem (mis. Pintu Air Manggarai di Jakarta Pusat).

### 10.3 Ringkasan error test per wilayah & bulan (`LinearRegression v2`)

- **Per wilayah:** `Jakarta Utara` (`MAE 0,78`, `RMSE 1,54`), `Jakarta Selatan` (`MAE 0,93`, `RMSE 1,14`), `Jakarta Timur` (`MAE 0,95`, `RMSE 1,22`), `Jakarta Barat` (`MAE 0,96`, `RMSE 1,84`), dan `Jakarta Pusat` (`MAE 1,19`, `RMSE 11,58` akibat fluktuasi ekstrem di Emplasemen PA Manggarai dengan `MAE 43,39`, `RMSE 143,85`).
- **Per bulan (Q4 2021):** Oktober (`MAE 0,80`), November (`MAE 0,93`), Desember (`MAE 1,04`).

### 10.4 Batasan yang wajib dicatat di laporan

1. **Skema split aktual:** memakai **split kronologis 70 / 15 / 15** (`train` `2018-01-01`–`2021-07-03`, `validation` `2021-07-04`–`2021-10-02`, `test` `2021-10-03`–`2021-12-31`) sesuai penyesuaian pembagian data terbaru; tahun 2019 tetap tidak ada pada sumber data.
2. **Penanganan outlier saat training:** `is_outlier == 1` (`> P99,9 = 65,52 m³`) dikecualikan dari `train`, dan fitur historis `v2` di-winsorize pada `P99,9` tanpa leakage (`rowsBetween(..., -1)`). `validation` dan `test` dievaluasi penuh apa adanya.
3. **Fitur `v2` untuk masa depan:** forecast jangka panjang di `PrediktorMasaDepan` memakai rata-rata historis per titik jika lag terbaru belum tersedia.
4. **Pemilihan metrik:** `LinearRegression v2` unggul di RMSE (`3,7404 m³`), sedangkan `RF v2` / `GBT v2` unggul di MAE (`0,746`–`0,755 m³`); jika operasional lebih memprioritaskan error harian median dibanding penalti lonjakan besar, `RF v2` atau `GBT v2` dapat dipilih.

## 11. Checklist & Troubleshooting

### 11.1 Checklist checkpoint Step 6

- [x] `df_v1` dan `df_v2` berhasil dimuat (`1.068.562` baris bersih, nama lokasi ternormalisasi).
- [x] Split kronologis **70 / 15 / 15** (`70,09%` / `15,03%` / `14,88%`) lolos `validasi_urutan_waktu`.
- [x] Baseline, LR, RF, dan GBT (`v1` & `v2`) selesai dievaluasi pada validation dan test.
- [x] Model terbaik (`linearregression_v2_20261121_170617`) tersimpan di `models/` beserta metadata dan metrik CSV.
- [x] Figure evaluasi tersimpan di `docs/figures/`.
- [x] Bagian 10 terisi berdasarkan angka nyata eksekusi.

### 11.2 Troubleshooting cepat

| Gejala | Penyebab umum | Solusi |
|--------|---------------|--------|
| `FileNotFoundError` dataset fitur | Step 4 belum jalan / path salah | Jalankan Step 3–4 atau set `SIPEKA_FEATURES_V1/V2` |
| `maxBins` error di RF/GBT | Kardinalitas kategori > `maxBins` | Notebook menghitung otomatis untuk `train` dan `train_full` |
| OOM / container restart | Grid tree terlalu berat untuk RAM terbatas | Pakai `local[2..4]`, `maxMemoryInMB=64`, `StorageLevel.MEMORY_AND_DISK` |
| `AnalysisException` path HDFS | Path lokal tanpa skema `file://` | Gunakan `path_ke_uri()` yang sudah disediakan |
| Prediksi `None` | Fitur null setelah join | Cek nama `titik_lokasi` dan ketersediaan lag untuk `v2` |